In [1]:
import os
import numpy as np
import xarray as xr

In [5]:
Dir = "/n/holylfs06/LABS/jacob_lab2/Lab/dzhang8/imi-gchp-test/imi-gchp-precomputedK"
sv_Dir = "/n/holylfs06/LABS/jacob_lab2/Lab/dzhang8/imi-gchp-test/output-gchp-stretch-soil"
CS_RES = 36
STRETCH_FACTOR = 10.
# Combine all state vectors
sv_ds_all = []
target_lats = []
target_lons = []
for ti in range(220):
    sv_fpath = f"{sv_Dir}/Test_Stretch_1day_T{ti+1:03d}/StateVector.nc"
    sv_ds = xr.open_dataset(sv_fpath, mask_and_scale=False)
    
    target_lats.append(sv_ds.attrs["TARGET_LAT"])
    target_lons.append(sv_ds.attrs["TARGET_LON"])
    sv_ds_all.append(sv_ds)
combined_ds = xr.concat(sv_ds_all, dim="target_face")
combined_ds['StateVector'] = combined_ds['StateVector'].transpose('time', 'target_face', 'nf', 'Ydim', 'Xdim')
combined_ds['TARGET_LAT'] = xr.DataArray(target_lats, dims=["target_face"], 
                                         attrs=dict(long_name='Target latitude',
                                                    units='degree_north'))
combined_ds['TARGET_LON'] = xr.DataArray(target_lons, dims=["target_face"], 
                                         attrs=dict(long_name='Target longitude',
                                                    units='degree_east'))
# add global attribites
combined_ds.attrs = {'CS_RES': CS_RES, 'STRETCH_FACTOR': STRETCH_FACTOR}
save_pth = f'{Dir}/StateVector_c{CS_RES}_s{STRETCH_FACTOR:.1f}_combined.nc'
combined_ds.to_netcdf(
            save_pth,
            encoding={
                v: {"zlib": True, "complevel": 1} for v in combined_ds.data_vars
            },
        )